# Формирование датасета по классам и добавление синтетики

Этот ноутбук собирает размеченные отзывы по каждому классу и дополняет редкие классы синтетическими примерами.

## Зачем нужен ноутбук

На вход подается файл с уже размеченными отзывами:

```text
data/labeled/wb_feedbacks_ChatGpt_markup/
chatgpt_labeled_reviews_mvp_for_training.csv
```

Если основной файл не найден, ноутбук может использовать fallback-файлы из той же папки с ChatGPT-разметкой.

Цель этапа — получить более сбалансированную корзинку отзывов по классам: реальные отзывы используются как основа, а для классов с недостаточным числом примеров догенерируются синтетические отзывы.

## Что делает ноутбук

1. Загружает размеченные отзывы с колонками:

```text
отзыв
labels
```

2. Раскладывает отзывы по MVP-классам.
3. Считает, сколько реальных примеров есть в каждом классе.
4. Если в классе меньше целевого количества примеров, генерирует недостающие синтетические отзывы.
5. Проверяет синтетические отзывы дополнительным LLM-проходом.
6. Сохраняет отдельный CSV-файл для каждого класса.

## Используемая модель

Для генерации синтетических отзывов используется модель:

```text
gpt-4.1
```

Модель вызывается через OpenAI API:

```python
MODEL = "gpt-4.1"
```

Для генерации используется повышенная температура:

```python
SYNTH_TEMPERATURE = 0.8
```

Это нужно, чтобы синтетические отзывы были более разнообразными.

Также включена LLM-валидация синтетики:

```python
USE_LLM_VALIDATION = True
```

То есть модель не только генерирует отзывы, но и отдельным проходом проверяет, подходят ли они под нужный класс.

Ссылка на документацию OpenAI по моделям:

```text
https://platform.openai.com/docs/models
```

## Выходные данные

Итоговая папка:

```text
data/labeled/wb_feedbacks_by_class_with_synthetic/
```

Внутри сохраняется по одному CSV-файлу на каждый класс.

Также сохраняется сводка:

```text
data/labeled/wb_feedbacks_by_class_with_synthetic/
_summary_by_class.csv
```

В ней указано, сколько в каждом классе:

```text
real_count       — реальных отзывов
synthetic_count  — синтетических отзывов
total_count      — всего отзывов
```

## Роль в пайплайне

```text
ChatGPT-размеченные отзывы
        ↓
generate_by_class_with_synthetic.ipynb
        ↓
real + synthetic отзывы по классам
        ↓
ChatGpt_markup_gpt5_from_synthetic.ipynb
        ↓
основной train-датасет
```

На этом этапе модель-классификатор еще не обучается.  
Ноутбук только подготавливает более сбалансированную корзинку примеров для последующей GPT-5-переразметки.

In [13]:
# В Google Colab раскомментируй, если зависимости еще не установлены:
# !pip install -q openai pandas tqdm pyarrow openpyxl

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os
import json
import time
import re
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm
from openai import OpenAI

## 1. Основные настройки

Проверь `PROJECT_ROOT`. Он должен указывать на папку `data`, внутри которой лежит `labeled`.

In [15]:
# ====== ПУТИ ======

PROJECT_ROOT = Path("/content/drive/MyDrive/MLops_project/project/data").resolve()
LABELED_DIR = PROJECT_ROOT / "labeled"

# Основной вариант: берем итоговый файл из предыдущего ноутбука ChatGpt-разметки.
INPUT_TRAIN_PATH = (
    LABELED_DIR
    / "wb_feedbacks_ChatGpt_markup"
    / "chatgpt_labeled_reviews_mvp_for_training.csv"
)

# Если итогового файла нет, можно читать отдельные файлы из папки разметки.
# В обычном сценарии этот список не понадобится.
FALLBACK_INPUT_FILES = [
    LABELED_DIR
    / "wb_feedbacks_ChatGpt_markup"
    / "chatgpt_mvp__labeled__wb_feedbacks_llm_labeled_from_sample__balanced_50_per_class_random_chunks_final.csv",
    LABELED_DIR
    / "wb_feedbacks_ChatGpt_markup"
    / "chatgpt_mvp__labeled__wb_feedbacks_llm_labeled_from_embeddings__extra_from_embedding_similarity_rare_classes_final.csv",
    LABELED_DIR
    / "wb_feedbacks_ChatGpt_markup"
    / "chatgpt_mvp__labeled__wb_feedbacks_llm_labeled_from_clusters__extra_from_hdbscan_candidate_clusters_final.csv",
]

OUTPUT_BY_CLASS_DIR = LABELED_DIR / "wb_feedbacks_by_class_with_synthetic"
OUTPUT_BY_CLASS_DIR.mkdir(parents=True, exist_ok=True)

SYNTH_PROGRESS_DIR = OUTPUT_BY_CLASS_DIR / "_synthetic_progress"
SYNTH_PROGRESS_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COLUMN = "отзыв"
LABELS_COLUMN = "labels"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("INPUT_TRAIN_PATH:", INPUT_TRAIN_PATH)
print("OUTPUT_BY_CLASS_DIR:", OUTPUT_BY_CLASS_DIR)

PROJECT_ROOT: /content/drive/MyDrive/MLops_project/project/data
INPUT_TRAIN_PATH: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_ChatGpt_markup/chatgpt_labeled_reviews_mvp_for_training.csv
OUTPUT_BY_CLASS_DIR: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic


In [16]:
ALLOWED_LABELS_MVP = [
    "Проблема с качеством товара",
    "Проблема с размером / посадкой",
    "Несоответствие карточке товара",
    "Проблема с комплектацией / упаковкой",
    "Проблема доставки / получения",
    "Цена / ценность",
    "Проблема с возвратом",
    "Положительный / нейтральный отзыв",
    "Другая проблема",
]

ALLOWED_LABELS_SET = set(ALLOWED_LABELS_MVP)

# Сколько итоговых примеров хотим получить в каждом файле класса.
# Для текущего датасета разумно начать с 200.
TARGET_PER_CLASS = 200

# Сколько синтетических примеров генерировать за один API-вызов.
# Если ответ будет обрезаться, уменьши до 10-15.
SYNTH_BATCH_SIZE = 20

# Используем ту же OpenAI API-логику, что в ноутбуке разметки.
# Можно заменить на модель, которой ты размечал данные.
MODEL = "gpt-4.1"

# Температура выше 0 нужна, чтобы синтетика была разнообразной.
SYNTH_TEMPERATURE = 0.8

# Второй LLM-проход для проверки синтетики.
# Лучше оставить True для качества. Если нужно дешевле/быстрее — поставь False.
USE_LLM_VALIDATION = True

MAX_COMPLETION_TOKENS = 6000
SLEEP_SECONDS = 0.5

import os
from openai import OpenAI

# Вариант 1: безопаснее для Colab
# Вставь ключ при запуске ячейки, он не будет храниться в ноутбуке
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Введите OPENAI_API_KEY: ")

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("OpenAI client initialized")

OpenAI client initialized


## 2. Вспомогательные функции

In [17]:
def read_csv_safely(path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "cp1251"]
    last_error = None

    for encoding in encodings:
        try:
            return pd.read_csv(path, encoding=encoding)
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Не удалось прочитать файл {path}. Последняя ошибка: {last_error}")


def labels_to_list(labels: Any) -> list[str]:
    if isinstance(labels, list):
        raw = labels
    elif pd.isna(labels):
        raw = []
    elif isinstance(labels, str):
        value = labels.strip()
        if not value:
            raw = []
        else:
            try:
                parsed = json.loads(value)
                raw = parsed if isinstance(parsed, list) else []
            except Exception:
                if " | " in value:
                    raw = [x.strip() for x in value.split(" | ")]
                elif value in ALLOWED_LABELS_SET:
                    raw = [value]
                else:
                    raw = []
    else:
        raw = []

    result = []
    for label in raw:
        label = str(label).strip()
        if label in ALLOWED_LABELS_SET and label not in result:
            result.append(label)

    return result


def normalize_labels(labels: Any) -> list[str]:
    result = labels_to_list(labels)

    if not result:
        result = ["Другая проблема"]

    neutral_label = "Положительный / нейтральный отзыв"

    # Если есть конкретная проблема, нейтральный класс убираем.
    if neutral_label in result and len(result) > 1:
        result = [label for label in result if label != neutral_label]

    if not result:
        result = ["Другая проблема"]

    return result


def labels_to_json(labels: Any) -> str:
    return json.dumps(normalize_labels(labels), ensure_ascii=False)


def safe_filename(label: str) -> str:
    name = label.lower()
    name = name.replace(" / ", "_")
    name = name.replace("/", "_")
    name = name.replace(" ", "_")
    name = re.sub(r"[^а-яa-z0-9_]+", "", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name + ".csv"


def append_jsonl(path: Path, rows: list[dict[str, Any]]) -> None:
    with path.open("a", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []

    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))

    return rows

## 3. Загрузка размеченных данных

Сначала пробуем прочитать `chatgpt_labeled_reviews_mvp_for_training.csv`. Если его нет, читаем отдельные CSV из `FALLBACK_INPUT_FILES`.

In [18]:
def load_labeled_data() -> pd.DataFrame:
    if INPUT_TRAIN_PATH.exists():
        print("Читаю итоговый train-файл:", INPUT_TRAIN_PATH)
        return read_csv_safely(INPUT_TRAIN_PATH)

    print("Итоговый train-файл не найден. Читаю fallback-файлы.")
    frames = []

    for path in FALLBACK_INPUT_FILES:
        if not path.exists():
            print("Пропускаю, файл не найден:", path)
            continue

        part = read_csv_safely(path)
        part["source_file"] = str(path)
        frames.append(part)
        print("Загружено:", path, "строк:", len(part))

    if not frames:
        raise FileNotFoundError(
            "Не найден ни INPUT_TRAIN_PATH, ни один файл из FALLBACK_INPUT_FILES. "
            "Проверь пути в настройках."
        )

    return pd.concat(frames, ignore_index=True)


base_df = load_labeled_data()

if TEXT_COLUMN not in base_df.columns:
    raise ValueError(f"Не найдена колонка {TEXT_COLUMN!r}. Колонки: {list(base_df.columns)}")

if LABELS_COLUMN not in base_df.columns:
    raise ValueError(f"Не найдена колонка {LABELS_COLUMN!r}. Колонки: {list(base_df.columns)}")

base_df = base_df.copy()
base_df[TEXT_COLUMN] = base_df[TEXT_COLUMN].fillna("").astype(str).str.strip()
base_df = base_df[base_df[TEXT_COLUMN].ne("")].copy()

base_df["labels_list"] = base_df[LABELS_COLUMN].apply(normalize_labels)
base_df[LABELS_COLUMN] = base_df["labels_list"].apply(lambda x: json.dumps(x, ensure_ascii=False))
base_df["source_type"] = "real"

# Убираем полные дубли отзывов, чтобы не раздувать датасет.
base_df = base_df.drop_duplicates(subset=[TEXT_COLUMN]).reset_index(drop=True)

print("Всего уникальных реальных отзывов:", len(base_df))
base_df.head()

Читаю итоговый train-файл: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_ChatGpt_markup/chatgpt_labeled_reviews_mvp_for_training.csv
Всего уникальных реальных отзывов: 1196


,отзыв,labels,labels_list,source_type
0,"Качество хорошее, но мне влики, отказ","[""Проблема с размером / посадкой""]",[Проблема с размером / посадкой],real
1,"Волшебно! Все супер, берите, не пожалеете)","[""Положительный / нейтральный отзыв""]",[Положительный / нейтральный отзыв],real
2,❤️❤️❤️❤️❤️ просто прелесть,"[""Положительный / нейтральный отзыв""]",[Положительный / нейтральный отзыв],real
3,Все пучком,"[""Положительный / нейтральный отзыв""]",[Положительный / нейтральный отзыв],real
4,Очень вкусноооо))) Спасибо большое!,"[""Положительный / нейтральный отзыв""]",[Положительный / нейтральный отзыв],real


In [19]:
stats_rows = []

for label in ALLOWED_LABELS_MVP:
    count = int(base_df["labels_list"].apply(lambda labels: label in labels).sum())
    stats_rows.append({
        "label": label,
        "real_count": count,
        "need_synthetic": max(0, TARGET_PER_CLASS - count),
    })

stats_df = pd.DataFrame(stats_rows).sort_values("real_count", ascending=False).reset_index(drop=True)
stats_df

,label,real_count,need_synthetic
0,Проблема с качеством товара,528,0
1,Проблема с комплектацией / упаковкой,392,0
2,Проблема с возвратом,238,0
3,Несоответствие карточке товара,210,0
4,Проблема с размером / посадкой,141,59
5,Положительный / нейтральный отзыв,85,115
6,Проблема доставки / получения,82,118
7,Цена / ценность,48,152
8,Другая проблема,20,180


## 4. Промпты и JSON-схемы для генерации синтетики

In [20]:
SYNTH_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "synthetic_marketplace_reviews",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "review": {"type": "string"},
                            "labels": {
                                "type": "array",
                                "items": {
                                    "type": "string",
                                    "enum": ALLOWED_LABELS_MVP,
                                },
                            },
                        },
                        "required": ["review", "labels"],
                    },
                }
            },
            "required": ["items"],
        },
    },
}

VALIDATION_RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "synthetic_reviews_validation",
        "strict": True,
        "schema": {
            "type": "object",
            "additionalProperties": False,
            "properties": {
                "items": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "additionalProperties": False,
                        "properties": {
                            "review": {"type": "string"},
                            "corrected_labels": {
                                "type": "array",
                                "items": {
                                    "type": "string",
                                    "enum": ALLOWED_LABELS_MVP,
                                },
                            },
                            "is_valid": {"type": "boolean"},
                            "reason": {"type": "string"},
                        },
                        "required": ["review", "corrected_labels", "is_valid", "reason"],
                    },
                }
            },
            "required": ["items"],
        },
    },
}

In [21]:
def build_synthetic_system_prompt() -> str:
    return f"""
Ты генерируешь синтетические отзывы покупателей о товарах на маркетплейсе для обучения multi-label классификатора.

Нужно генерировать реалистичные русскоязычные отзывы.

Разрешенные labels:
{json.dumps(ALLOWED_LABELS_MVP, ensure_ascii=False, indent=2)}

Описание классов:

1. "Проблема с качеством товара"
Брак, дефект, поломка, повреждение товара, плохой материал, запах, плохой пошив, царапины, сколы, трещины, хлипкость, подделка.

2. "Проблема с размером / посадкой"
Маломерит, большемерит, не подошел размер, плохо сидит, слишком узкое/широкое/короткое/длинное, не подходит по форме или совместимости.

3. "Несоответствие карточке товара"
Товар отличается от фото, описания, характеристик, состава, цвета, количества, модели или заявленного эффекта.

4. "Проблема с комплектацией / упаковкой"
Неполный комплект, не хватает деталей, пришел не тот комплект, нет инструкции, упаковка повреждена, коробка мятая, плохо упаковано.

5. "Проблема доставки / получения"
Долгая доставка, задержка, курьер, пункт выдачи, потеря заказа, отмена заказа, проблема с получением.

6. "Цена / ценность"
Дорого, не стоит своих денег, цена завышена, качество не соответствует цене.

7. "Проблема с возвратом"
Жалобы на возврат товара или денег: не приняли возврат, отказали в возврате, сложно оформить возврат, долго возвращают деньги, проблемы с компенсацией после покупки.

8. "Положительный / нейтральный отзыв"
Положительный, нейтральный или информационный отзыв без явной жалобы.

9. "Другая проблема"
Есть негатив или жалоба, но она не подходит к классам выше.

Правила:
- Один отзыв может иметь один или несколько labels.
- Если есть конкретная проблема, НЕ добавляй "Положительный / нейтральный отзыв".
- Если конкретной проблемы нет, используй только "Положительный / нейтральный отзыв".
- Названия labels должны полностью совпадать со списком.
- Не добавляй evidence, confidence, markdown, пояснения или текст вне JSON.
- Отзывы должны быть похожи на реальные маркетплейс-отзывы: короткие, бытовые, иногда с ошибками и разговорными фразами.
- Не пиши слишком литературно.
- Не используй конкретные бренды.
- Не копируй дословно реальные примеры.
- Не делай все отзывы одинаковыми по структуре.
- В синтетике должны быть разные товары: одежда, косметика, бытовые товары, электроника, аксессуары, товары для дома, детские товары, обувь.
""".strip()


def get_subtypes_for_label(target_label: str) -> list[str]:
    subtypes = {
        "Проблема с качеством товара": [
            "товар сломался быстро",
            "плохой материал",
            "есть дефект или брак",
            "неприятный запах",
            "хлипкая конструкция",
            "плохой пошив или сборка",
        ],
        "Проблема с размером / посадкой": [
            "маломерит",
            "большемерит",
            "плохо сидит",
            "не подошло по форме",
            "слишком узкое или широкое",
            "не совпал размер с таблицей",
        ],
        "Несоответствие карточке товара": [
            "цвет отличается от фото",
            "материал не такой, как в описании",
            "пришел другой товар",
            "характеристики не совпадают",
            "ожидаемый эффект не соответствует описанию",
        ],
        "Проблема с комплектацией / упаковкой": [
            "не хватает детали",
            "пришел неполный комплект",
            "упаковка мятая или порвана",
            "плохо упаковано",
            "нет инструкции",
        ],
        "Проблема доставки / получения": [
            "долго доставляли",
            "перенесли срок доставки",
            "заказ потерялся",
            "проблемы в пункте выдачи",
            "курьер доставил неудобно",
        ],
        "Цена / ценность": [
            "цена завышена",
            "товар не стоит своих денег",
            "качество не соответствует цене",
            "за такую цену ожидали лучше",
            "слишком маленький объем за такую цену",
        ],
        "Проблема с возвратом": [
            "долго возвращают деньги",
            "отказали в возврате",
            "сложно оформить возврат",
            "списали деньги за возврат",
            "поддержка не решает вопрос возврата",
        ],
        "Положительный / нейтральный отзыв": [
            "товар понравился",
            "обычный товар без жалоб",
            "соответствует описанию",
            "быстрая доставка без проблем",
            "еще не пробовали, но внешне нормально",
        ],
        "Другая проблема": [
            "неудобно пользоваться",
            "непонятная инструкция",
            "не подошло по личным причинам",
            "странные ощущения от использования",
            "есть негатив, но без конкретной категории",
        ],
    }
    return subtypes.get(target_label, [])


def sample_examples_for_prompt(df: pd.DataFrame, target_label: str, n: int = 12) -> list[dict[str, Any]]:
    part = df[df["labels_list"].apply(lambda labels: target_label in labels)].copy()

    if part.empty:
        return []

    part = part.sample(n=min(n, len(part)), random_state=42)

    examples = []
    for _, row in part.iterrows():
        examples.append({
            "review": str(row[TEXT_COLUMN]),
            "labels": row["labels_list"],
        })

    return examples


def build_synthetic_user_prompt(target_label: str, n: int, real_examples: list[dict[str, Any]]) -> str:
    neutral_label = "Положительный / нейтральный отзыв"
    subtypes = get_subtypes_for_label(target_label)

    if target_label == neutral_label:
        distribution = f"""
Сгенерируй {n} отзывов для целевого класса "{target_label}".

Все отзывы должны быть без конкретной жалобы.
Для каждого отзыва labels должны быть строго:
["{target_label}"]
"""
    else:
        single_count = max(1, int(n * 0.7))
        multi_count = n - single_count
        distribution = f"""
Сгенерируй {n} отзывов для целевого класса "{target_label}".

Распределение:
- {single_count} отзывов должны быть single-label только с классом "{target_label}".
- {multi_count} отзывов должны быть multi-label: "{target_label}" + один дополнительный проблемный класс.

Не добавляй "Положительный / нейтральный отзыв", если есть проблема.
"""

    return f"""
Нужно догенерировать качественные синтетические отзывы для редкого класса.

Целевой класс:
"{target_label}"

{distribution}

Подтипы и ситуации, которые нужно разнообразно покрыть:
{json.dumps(subtypes, ensure_ascii=False, indent=2)}

Реальные примеры этого класса, на стиль которых можно ориентироваться, но нельзя копировать дословно:
{json.dumps(real_examples, ensure_ascii=False, indent=2)}

Верни только JSON строго такого вида:
{{
  "items": [
    {{
      "review": "...",
      "labels": ["..."]
    }}
  ]
}}
""".strip()

## 5. Генерация и проверка синтетики

Результаты каждого класса сохраняются в `_synthetic_progress/*.jsonl`. Если ноутбук остановится, повторный запуск не потеряет уже сгенерированные примеры.

In [22]:
def validate_synthetic_item_locally(item: dict[str, Any], target_label: str) -> dict[str, Any] | None:
    review = str(item.get("review", "")).strip()
    labels = normalize_labels(item.get("labels", []))

    if not review:
        return None

    if len(review) < 5:
        return None

    if target_label not in labels:
        return None

    neutral_label = "Положительный / нейтральный отзыв"
    if neutral_label in labels and len(labels) > 1:
        labels = [label for label in labels if label != neutral_label]

    return {
        TEXT_COLUMN: review,
        LABELS_COLUMN: json.dumps(labels, ensure_ascii=False),
        "labels_list": labels,
        "source_type": "synthetic",
        "target_label": target_label,
    }


def call_openai_json(messages: list[dict[str, str]], response_format: dict[str, Any], temperature: float, max_retries: int = 3) -> dict[str, Any]:
    last_error = None

    for attempt in range(1, max_retries + 1):
        try:
            response = client.chat.completions.create(
                model=MODEL,
                messages=messages,
                response_format=response_format,
                temperature=temperature,
                timeout=300,
                max_completion_tokens=MAX_COMPLETION_TOKENS,
            )

            finish_reason = response.choices[0].finish_reason
            content = response.choices[0].message.content or ""

            if finish_reason != "stop":
                raise RuntimeError(f"Ответ модели был обрезан или остановлен не штатно: finish_reason={finish_reason}")

            return json.loads(content)

        except Exception as e:
            last_error = e
            wait = 2 ** attempt
            print(f"Ошибка OpenAI API на попытке {attempt}/{max_retries}: {e}")
            print(f"Ждем {wait} сек. и пробуем снова...")
            time.sleep(wait)

    raise RuntimeError(f"Не удалось получить корректный JSON от OpenAI API. Последняя ошибка: {last_error}")


def generate_synthetic_batch(target_label: str, n: int, real_examples: list[dict[str, Any]]) -> list[dict[str, Any]]:
    print(f"Генерация: {target_label} | нужно: {n}")

    parsed = call_openai_json(
        messages=[
            {"role": "system", "content": build_synthetic_system_prompt()},
            {"role": "user", "content": build_synthetic_user_prompt(target_label, n, real_examples)},
        ],
        response_format=SYNTH_RESPONSE_FORMAT,
        temperature=SYNTH_TEMPERATURE,
    )

    items = parsed.get("items", [])
    valid_items = []

    for item in items:
        valid_item = validate_synthetic_item_locally(item, target_label)
        if valid_item is not None:
            valid_items.append(valid_item)

    return valid_items


def build_validation_prompt(target_label: str, items: list[dict[str, Any]]) -> str:
    compact_items = [
        {
            "review": row[TEXT_COLUMN],
            "labels": normalize_labels(row[LABELS_COLUMN]),
        }
        for row in items
    ]

    return f"""
Ты проверяешь синтетические отзывы для обучения multi-label классификатора отзывов маркетплейса.

Целевой класс: "{target_label}"

Разрешенные labels:
{json.dumps(ALLOWED_LABELS_MVP, ensure_ascii=False, indent=2)}

Для каждого объекта проверь:
1. Реалистичен ли отзыв для маркетплейса.
2. Соответствуют ли labels тексту отзыва.
3. Нет ли лишних labels.
4. Нет ли пропущенных labels.
5. Не выбран ли "Положительный / нейтральный отзыв" вместе с проблемным классом.
6. Есть ли в отзыве целевой класс "{target_label}".
7. Не слишком ли отзыв шаблонный или искусственный.

Если отзыв плохой, слишком искусственный или не содержит целевой класс — is_valid=false.
Если labels можно исправить и отзыв хороший — is_valid=true и corrected_labels должны содержать правильные labels.

Верни JSON строго по схеме.

Отзывы для проверки:
{json.dumps(compact_items, ensure_ascii=False, indent=2)}
""".strip()


def validate_synthetic_batch_with_llm(target_label: str, items: list[dict[str, Any]]) -> list[dict[str, Any]]:
    if not items:
        return []

    print(f"LLM-проверка синтетики: {target_label} | кандидатов: {len(items)}")

    parsed = call_openai_json(
        messages=[
            {"role": "system", "content": "Ты строгий валидатор синтетических данных для классификатора отзывов. Возвращай только JSON."},
            {"role": "user", "content": build_validation_prompt(target_label, items)},
        ],
        response_format=VALIDATION_RESPONSE_FORMAT,
        temperature=0,
    )

    validated_raw = parsed.get("items", [])
    result = []

    for item in validated_raw:
        if not item.get("is_valid", False):
            continue

        corrected = validate_synthetic_item_locally(
            {
                "review": item.get("review", ""),
                "labels": item.get("corrected_labels", []),
            },
            target_label=target_label,
        )

        if corrected is not None:
            result.append(corrected)

    return result

In [23]:
def progress_path_for_label(target_label: str) -> Path:
    return SYNTH_PROGRESS_DIR / safe_filename(target_label).replace(".csv", ".jsonl")


def get_existing_synthetic_for_label(target_label: str) -> pd.DataFrame:
    path = progress_path_for_label(target_label)
    rows = read_jsonl(path)

    if not rows:
        return pd.DataFrame(columns=[TEXT_COLUMN, LABELS_COLUMN, "labels_list", "source_type", "target_label"])

    df = pd.DataFrame(rows)
    df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str).str.strip()
    df = df[df[TEXT_COLUMN].ne("")].copy()
    df["labels_list"] = df[LABELS_COLUMN].apply(normalize_labels)
    df[LABELS_COLUMN] = df["labels_list"].apply(lambda x: json.dumps(x, ensure_ascii=False))
    df["source_type"] = "synthetic"
    df["target_label"] = target_label

    return df


def save_synthetic_progress(target_label: str, rows: list[dict[str, Any]]) -> None:
    path = progress_path_for_label(target_label)

    rows_to_save = []
    for row in rows:
        rows_to_save.append({
            TEXT_COLUMN: row[TEXT_COLUMN],
            LABELS_COLUMN: row[LABELS_COLUMN],
            "source_type": "synthetic",
            "target_label": target_label,
        })

    append_jsonl(path, rows_to_save)


def deduplicate_generated_rows(
    generated_rows: list[dict[str, Any]],
    existing_texts: set[str],
) -> list[dict[str, Any]]:
    result = []

    for row in generated_rows:
        text = str(row[TEXT_COLUMN]).strip()
        if text and text not in existing_texts:
            result.append(row)
            existing_texts.add(text)

    return result

## 6. Основной цикл: создаем по одному файлу на каждый класс

В multi-label задаче один реальный отзыв может попасть в несколько файлов классов. Это нормально: файл класса содержит все отзывы, где этот класс присутствует.

In [24]:
summary_rows = []

for target_label in tqdm(ALLOWED_LABELS_MVP, desc="Файлы по классам"):
    output_path = OUTPUT_BY_CLASS_DIR / safe_filename(target_label)

    real_part = base_df[
        base_df["labels_list"].apply(lambda labels: target_label in labels)
    ].copy()

    real_part = real_part[[TEXT_COLUMN, LABELS_COLUMN, "labels_list", "source_type"]].copy()
    real_count = len(real_part)

    synthetic_existing = get_existing_synthetic_for_label(target_label)

    need_total = max(0, TARGET_PER_CLASS - real_count)
    already_synth_count = len(synthetic_existing)
    need_to_generate = max(0, need_total - already_synth_count)

    print("\n" + "=" * 100)
    print(target_label)
    print("Реальных примеров:", real_count)
    print("Уже есть синтетики:", already_synth_count)
    print("Нужно догенерировать:", need_to_generate)

    if need_to_generate > 0:
        real_examples = sample_examples_for_prompt(base_df, target_label, n=12)
        generated_total = []

        while len(generated_total) < need_to_generate:
            batch_n = min(SYNTH_BATCH_SIZE, need_to_generate - len(generated_total))

            candidates = generate_synthetic_batch(
                target_label=target_label,
                n=batch_n,
                real_examples=real_examples,
            )

            if USE_LLM_VALIDATION:
                candidates = validate_synthetic_batch_with_llm(target_label, candidates)

            existing_texts = set(real_part[TEXT_COLUMN].astype(str).str.strip().tolist())
            existing_texts.update(synthetic_existing[TEXT_COLUMN].astype(str).str.strip().tolist())
            existing_texts.update(row[TEXT_COLUMN].strip() for row in generated_total)

            unique_batch = deduplicate_generated_rows(candidates, existing_texts)

            if unique_batch:
                save_synthetic_progress(target_label, unique_batch)
                generated_total.extend(unique_batch)

            print(f"Сгенерировано новых уникальных: {len(generated_total)} / {need_to_generate}")

            # Если модель вернула 0 валидных примеров, уменьшаем batch и продолжаем.
            if not unique_batch:
                print("Валидных уникальных примеров не получено. Повторяем генерацию.")

            time.sleep(SLEEP_SECONDS)

        synthetic_existing = get_existing_synthetic_for_label(target_label)

    if synthetic_existing.empty:
        final_part = real_part.copy()
    else:
        synthetic_part = synthetic_existing[[TEXT_COLUMN, LABELS_COLUMN, "labels_list", "source_type"]].copy()
        final_part = pd.concat([real_part, synthetic_part], ignore_index=True)

    final_part = final_part.drop_duplicates(subset=[TEXT_COLUMN]).reset_index(drop=True)

    output_df = final_part[[TEXT_COLUMN, LABELS_COLUMN, "source_type"]].copy()
    output_df.to_csv(output_path, index=False, encoding="utf-8-sig")

    summary_rows.append({
        "label": target_label,
        "real_count": real_count,
        "synthetic_count": int((output_df["source_type"] == "synthetic").sum()),
        "total_count": len(output_df),
        "need_synthetic_initially": need_total,
        "output_file": str(output_path),
    })

    print("Сохранено:", output_path)
    print("Итого строк:", len(output_df))

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_BY_CLASS_DIR / "_summary_by_class.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\nГотово.")
print("Сводка сохранена:", summary_path)
summary_df

Файлы по классам:   0%|          | 0/9 [00:00<?, ?it/s]


Проблема с качеством товара
Реальных примеров: 528
Уже есть синтетики: 0
Нужно догенерировать: 0
Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic/проблема_с_качеством_товара.csv
Итого строк: 528

Проблема с размером / посадкой
Реальных примеров: 141
Уже есть синтетики: 40
Нужно догенерировать: 19
Генерация: Проблема с размером / посадкой | нужно: 19
LLM-проверка синтетики: Проблема с размером / посадкой | кандидатов: 19
Сгенерировано новых уникальных: 19 / 19
Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic/проблема_с_размером_посадкой.csv
Итого строк: 200

Несоответствие карточке товара
Реальных примеров: 210
Уже есть синтетики: 0
Нужно догенерировать: 0
Сохранено: /content/drive/MyDrive/MLops_project/project/data/labeled/wb_feedbacks_by_class_with_synthetic/несоответствие_карточке_товара.csv
Итого строк: 210

Проблема с комплектацией / упаковкой
Реальных примеров: 392
Уж

,label,real_count,synthetic_count,total_count,need_synthetic_initially,output_file
0,Проблема с качеством товара,528,0,528,0,/content/drive/MyDrive/MLops_project/project/d...
1,Проблема с размером / посадкой,141,59,200,59,/content/drive/MyDrive/MLops_project/project/d...
2,Несоответствие карточке товара,210,0,210,0,/content/drive/MyDrive/MLops_project/project/d...
3,Проблема с комплектацией / упаковкой,392,0,392,0,/content/drive/MyDrive/MLops_project/project/d...
4,Проблема доставки / получения,82,118,200,118,/content/drive/MyDrive/MLops_project/project/d...
5,Цена / ценность,48,152,200,152,/content/drive/MyDrive/MLops_project/project/d...
6,Проблема с возвратом,238,0,238,0,/content/drive/MyDrive/MLops_project/project/d...
7,Положительный / нейтральный отзыв,85,115,200,115,/content/drive/MyDrive/MLops_project/project/d...
8,Другая проблема,20,180,200,180,/content/drive/MyDrive/MLops_project/project/d...


## 7. Проверка сохраненных файлов

In [25]:
summary_df = read_csv_safely(OUTPUT_BY_CLASS_DIR / "_summary_by_class.csv")
summary_df

,label,real_count,synthetic_count,total_count,need_synthetic_initially,output_file
0,Проблема с качеством товара,528,0,528,0,/content/drive/MyDrive/MLops_project/project/d...
1,Проблема с размером / посадкой,141,59,200,59,/content/drive/MyDrive/MLops_project/project/d...
2,Несоответствие карточке товара,210,0,210,0,/content/drive/MyDrive/MLops_project/project/d...
3,Проблема с комплектацией / упаковкой,392,0,392,0,/content/drive/MyDrive/MLops_project/project/d...
4,Проблема доставки / получения,82,118,200,118,/content/drive/MyDrive/MLops_project/project/d...
5,Цена / ценность,48,152,200,152,/content/drive/MyDrive/MLops_project/project/d...
6,Проблема с возвратом,238,0,238,0,/content/drive/MyDrive/MLops_project/project/d...
7,Положительный / нейтральный отзыв,85,115,200,115,/content/drive/MyDrive/MLops_project/project/d...
8,Другая проблема,20,180,200,180,/content/drive/MyDrive/MLops_project/project/d...


In [26]:
for _, row in summary_df.iterrows():
    print(
        f"{row['label']}: "
        f"real={row['real_count']}, "
        f"synthetic={row['synthetic_count']}, "
        f"total={row['total_count']}"
    )

Проблема с качеством товара: real=528, synthetic=0, total=528
Проблема с размером / посадкой: real=141, synthetic=59, total=200
Несоответствие карточке товара: real=210, synthetic=0, total=210
Проблема с комплектацией / упаковкой: real=392, synthetic=0, total=392
Проблема доставки / получения: real=82, synthetic=118, total=200
Цена / ценность: real=48, synthetic=152, total=200
Проблема с возвратом: real=238, synthetic=0, total=238
Положительный / нейтральный отзыв: real=85, synthetic=115, total=200
Другая проблема: real=20, synthetic=180, total=200


In [27]:
# Посмотреть несколько примеров из каждого файла класса.

for label in ALLOWED_LABELS_MVP:
    path = OUTPUT_BY_CLASS_DIR / safe_filename(label)
    if not path.exists():
        continue

    part = read_csv_safely(path)

    print("\n" + "=" * 100)
    print(label, "| строк:", len(part), "| файл:", path.name)
    print("=" * 100)

    examples = part.sample(n=min(5, len(part)), random_state=42)
    for _, row in examples.iterrows():
        print("-", row[TEXT_COLUMN])
        print("  labels:", row[LABELS_COLUMN])
        print("  source_type:", row["source_type"])


Проблема с качеством товара | строк: 528 | файл: проблема_с_качеством_товара.csv
- Не делайте глупостей девочки! Ваши волосы станут паклей или мочалкой!
  labels: ["Проблема с качеством товара"]
  source_type: real
- Отказ в центре выдачи. По всем признакам бу!!!! Включили, работает, поднесли к руке с упором, начал дрибезжать. На любой скорости итог один и тот же. Видимо вернули по браку. Но почему-то продавец решил продавать его дальше!
  labels: ["Проблема с качеством товара"]
  source_type: real
- Пришли сломанные
  labels: ["Проблема с качеством товара"]
  source_type: real
- Зеркало симпатичное, но пришло разбитое. К сожалению, упаковка не очень прочная. За возврат 50 рублей сняли.
  labels: ["Проблема с качеством товара", "Проблема с комплектацией / упаковкой", "Проблема с возвратом"]
  source_type: real
- Не понравился,хлипкий нитки торчат,возврат
  labels: ["Проблема с качеством товара"]
  source_type: real

Проблема с размером / посадкой | строк: 200 | файл: проблема_с_размер

## 8. Как использовать дальше

Для обучения классификатора лучше делать так:

```text
train = реальные train-примеры + синтетика
validation = только реальные примеры
test = только реальные примеры
```

Не добавляй синтетику в validation/test, иначе метрики будут завышены.